# VisionGym LoRA / QLoRA Fine-tuning
동일한 synthetic train split으로 Qwen3-VL-2B-Instruct를 LoRA fine-tuning하고 동일 ID/OOD benchmark에서 비교합니다.

In [ ]:
# Colab GPU용 학습 의존성을 설치합니다.
!git clone -q https://github.com/oosuhada/visiongym.git /content/visiongym || true
%cd /content/visiongym
!git pull -q
!pip install -q -e '.[train]'

In [ ]:
# 기존 generated dataset이 없다면 CPU에서 먼저 생성합니다.
from pathlib import Path
import subprocess
if not Path('data/generated/train.jsonl').exists():
    subprocess.run(['visiongym', 'generate', '--config', 'configs/dataset.yaml', '--output', 'data/generated'], check=True)
!visiongym prepare-sft --dataset data/generated/train.jsonl --output data/generated/train_sft.jsonl

In [ ]:
# configs/training.yaml의 max_samples, lora_r, include_tasks, curriculum을 바꿔 ablation을 만들 수 있습니다.
!visiongym train-lora --config configs/training.yaml

In [ ]:
# Fine-tuned LoRA adapter를 동일 benchmark에 적용합니다.
!visiongym infer --dataset data/generated/benchmark.jsonl --output outputs/lora-direct.jsonl --model Qwen/Qwen3-VL-2B-Instruct --adapter checkpoints/visiongym-qwen3-vl-2b-lora --prompt-mode direct --load-in-4bit --batch-size 8
!visiongym evaluate --dataset data/generated/benchmark.jsonl --predictions outputs/lora-direct.jsonl --output reports/lora-direct --model Qwen/Qwen3-VL-2B-Instruct+VisionGym-LoRA --prompt-mode direct
!visiongym report --metrics reports/lora-direct/metrics.json --output reports/lora-direct

In [ ]:
# Baseline notebook 결과가 같은 runtime/Drive에 있으면 전후 수치를 한 표로 비교합니다.
from pathlib import Path
if Path('reports/base-direct/metrics.json').exists():
    !visiongym compare reports/base-direct/metrics.json reports/lora-direct/metrics.json --output reports/base-vs-lora.csv
else:
    print('Run notebooks/baseline.ipynb first to create the base comparison metrics.')